# Batch download + clean (spot monthly klines)

Символы: `BTCUSDT`, `ETHUSDT`, `BNBUSDT`, `XRPUSDT`, `DOGEUSDT`  
Таймфреймы: `1d`, `4h`  
Период: от самого первого доступного месяца до `31 марта 2026` включительно (`end_month=2026-03`).

Ячейки разделены так, чтобы можно было продолжить с места остановки.

In [7]:
from pathlib import Path
import sys
import pandas as pd

# make local modules importable from both repo root and data/ cwd
cwd = Path.cwd()
if (cwd / 'binance_monthly_downloader.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'data' / 'binance_monthly_downloader.py').exists():
    sys.path.insert(0, str(cwd / 'data'))

from binance_monthly_downloader import (
    download_and_save_monthly_klines_raw,
    infer_active_name,
    make_raw_output_path,
)
from klines_metrics_cleaner import process_raw_csv_to_features_csv, make_processed_output_path


In [8]:
# Config
SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]
INTERVALS = ["1d", "4h"]
MARKET = "spot"

# Берем с самого раннего доступного месяца
START_MONTH = None
# До 31 марта 2026 включительно => месячный файл 2026-03
END_MONTH = "2026-03"

RAW_DIR = Path("raw")
ZIP_CACHE_DIR = RAW_DIR / "_zip_cache"
PROCESSED_DIR = Path(".")

# Флаги для удобного рестарта
FORCE_REDOWNLOAD = False
FORCE_RECLEAN = False


In [9]:
# План работ (symbol x interval)
tasks = []
for symbol in SYMBOLS:
    active = infer_active_name(symbol)
    for interval in INTERVALS:
        raw_csv = make_raw_output_path(
            symbol=symbol, interval=interval, raw_dir=RAW_DIR, active_name=active
        )
        cleaned_csv = make_processed_output_path(active=active, interval=interval, output_dir=PROCESSED_DIR)
        tasks.append({
            "symbol": symbol,
            "active": active,
            "interval": interval,
            "raw_csv": str(raw_csv),
            "cleaned_csv": str(cleaned_csv),
        })

plan_df = pd.DataFrame(tasks)
plan_df


,symbol,active,interval,raw_csv,cleaned_csv
0,BTCUSDT,btc,1d,raw\btc-1d-RAW.csv,btc-1d.csv
1,BTCUSDT,btc,4h,raw\btc-4h-RAW.csv,btc-4h.csv
2,ETHUSDT,eth,1d,raw\eth-1d-RAW.csv,eth-1d.csv
3,ETHUSDT,eth,4h,raw\eth-4h-RAW.csv,eth-4h.csv
4,BNBUSDT,bnb,1d,raw\bnb-1d-RAW.csv,bnb-1d.csv
5,BNBUSDT,bnb,4h,raw\bnb-4h-RAW.csv,bnb-4h.csv
6,XRPUSDT,xrp,1d,raw\xrp-1d-RAW.csv,xrp-1d.csv
7,XRPUSDT,xrp,4h,raw\xrp-4h-RAW.csv,xrp-4h.csv
8,DOGEUSDT,doge,1d,raw\doge-1d-RAW.csv,doge-1d.csv
9,DOGEUSDT,doge,4h,raw\doge-4h-RAW.csv,doge-4h.csv


## 1) Обкачка raw CSV
Если ноутбук оборвется, просто запускай эту ячейку снова: уже готовые файлы будут пропущены (если `FORCE_REDOWNLOAD=False`).

In [10]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
ZIP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

download_rows = []
for t in tasks:
    symbol = t["symbol"]
    active = t["active"]
    interval = t["interval"]
    raw_path = Path(t["raw_csv"])

    if raw_path.exists() and not FORCE_REDOWNLOAD:
        print(f"[SKIP] raw exists: {raw_path}")
        download_rows.append({**t, "status": "skip_exists"})
        continue

    print(f"[DOWNLOAD] {symbol} {interval} -> {raw_path}")
    out = download_and_save_monthly_klines_raw(
        symbol=symbol,
        interval=interval,
        market=MARKET,
        start_month=START_MONTH,
        end_month=END_MONTH,
        raw_dir=RAW_DIR,
        cache_dir=ZIP_CACHE_DIR / f"{symbol.lower()}-{interval}",
        active_name=active,
    )
    download_rows.append({**t, "status": "downloaded", "saved": str(out)})

download_log = pd.DataFrame(download_rows)
download_log


[DOWNLOAD] BTCUSDT 1d -> raw\btc-1d-RAW.csv
Processed 50/104 monthly files...
Processed 100/104 monthly files...
[DOWNLOAD] BTCUSDT 4h -> raw\btc-4h-RAW.csv
Processed 50/104 monthly files...
Processed 100/104 monthly files...
[DOWNLOAD] ETHUSDT 1d -> raw\eth-1d-RAW.csv
Processed 50/104 monthly files...
Processed 100/104 monthly files...
[DOWNLOAD] ETHUSDT 4h -> raw\eth-4h-RAW.csv
Processed 50/103 monthly files...
Processed 100/103 monthly files...
[DOWNLOAD] BNBUSDT 1d -> raw\bnb-1d-RAW.csv
Processed 50/101 monthly files...
Processed 100/101 monthly files...
[DOWNLOAD] BNBUSDT 4h -> raw\bnb-4h-RAW.csv
Processed 50/101 monthly files...
Processed 100/101 monthly files...
[DOWNLOAD] XRPUSDT 1d -> raw\xrp-1d-RAW.csv
Processed 50/95 monthly files...
[DOWNLOAD] XRPUSDT 4h -> raw\xrp-4h-RAW.csv
Processed 50/93 monthly files...
[DOWNLOAD] DOGEUSDT 1d -> raw\doge-1d-RAW.csv
Processed 50/81 monthly files...
[DOWNLOAD] DOGEUSDT 4h -> raw\doge-4h-RAW.csv
Processed 50/81 monthly files...


,symbol,active,interval,raw_csv,cleaned_csv,status,saved
0,BTCUSDT,btc,1d,raw\btc-1d-RAW.csv,btc-1d.csv,downloaded,raw\btc-1d-RAW.csv
1,BTCUSDT,btc,4h,raw\btc-4h-RAW.csv,btc-4h.csv,downloaded,raw\btc-4h-RAW.csv
2,ETHUSDT,eth,1d,raw\eth-1d-RAW.csv,eth-1d.csv,downloaded,raw\eth-1d-RAW.csv
3,ETHUSDT,eth,4h,raw\eth-4h-RAW.csv,eth-4h.csv,downloaded,raw\eth-4h-RAW.csv
4,BNBUSDT,bnb,1d,raw\bnb-1d-RAW.csv,bnb-1d.csv,downloaded,raw\bnb-1d-RAW.csv
5,BNBUSDT,bnb,4h,raw\bnb-4h-RAW.csv,bnb-4h.csv,downloaded,raw\bnb-4h-RAW.csv
6,XRPUSDT,xrp,1d,raw\xrp-1d-RAW.csv,xrp-1d.csv,downloaded,raw\xrp-1d-RAW.csv
7,XRPUSDT,xrp,4h,raw\xrp-4h-RAW.csv,xrp-4h.csv,downloaded,raw\xrp-4h-RAW.csv
8,DOGEUSDT,doge,1d,raw\doge-1d-RAW.csv,doge-1d.csv,downloaded,raw\doge-1d-RAW.csv
9,DOGEUSDT,doge,4h,raw\doge-4h-RAW.csv,doge-4h.csv,downloaded,raw\doge-4h-RAW.csv


## 2) Очистка + метрики
Снова можно безопасно перезапускать: готовые `ACTIVE-TIMEFRAME.csv` будут пропущены (если `FORCE_RECLEAN=False`).

In [11]:
clean_rows = []
for t in tasks:
    active = t["active"]
    interval = t["interval"]
    raw_csv = Path(t["raw_csv"])
    cleaned_csv = Path(t["cleaned_csv"])

    if not raw_csv.exists():
        print(f"[MISS RAW] {raw_csv}")
        clean_rows.append({**t, "status": "missing_raw"})
        continue

    if cleaned_csv.exists() and not FORCE_RECLEAN:
        print(f"[SKIP] clean exists: {cleaned_csv}")
        clean_rows.append({**t, "status": "skip_exists"})
        continue

    print(f"[CLEAN] {raw_csv.name} -> {cleaned_csv.name}")
    out_path, stats = process_raw_csv_to_features_csv(
        raw_csv=raw_csv,
        active=active,
        interval=interval,
        output_dir=PROCESSED_DIR,
    )

    clean_rows.append({
        **t,
        "status": "cleaned",
        "saved": str(out_path),
        "rows_before": stats.rows_before,
        "rows_after_dropna_ohlc": stats.rows_after_dropna_ohlc,
        "rows_after_feature_dropna": stats.rows_after_feature_dropna,
        "dropped_rows_ohlc_na": stats.dropped_rows_ohlc_na,
        "dropped_rows_feature_na": stats.dropped_rows_feature_na,
    })

clean_log = pd.DataFrame(clean_rows)
clean_log


[CLEAN] btc-1d-RAW.csv -> btc-1d.csv
[CLEAN] btc-4h-RAW.csv -> btc-4h.csv
[CLEAN] eth-1d-RAW.csv -> eth-1d.csv
[CLEAN] eth-4h-RAW.csv -> eth-4h.csv
[CLEAN] bnb-1d-RAW.csv -> bnb-1d.csv
[CLEAN] bnb-4h-RAW.csv -> bnb-4h.csv
[CLEAN] xrp-1d-RAW.csv -> xrp-1d.csv
[CLEAN] xrp-4h-RAW.csv -> xrp-4h.csv
[CLEAN] doge-1d-RAW.csv -> doge-1d.csv
[CLEAN] doge-4h-RAW.csv -> doge-4h.csv


,symbol,active,interval,raw_csv,cleaned_csv,status,saved,rows_before,rows_after_dropna_ohlc,rows_after_feature_dropna,dropped_rows_ohlc_na,dropped_rows_feature_na
0,BTCUSDT,btc,1d,raw\btc-1d-RAW.csv,btc-1d.csv,cleaned,btc-1d.csv,3149,3149,3119,0,30
1,BTCUSDT,btc,4h,raw\btc-4h-RAW.csv,btc-4h.csv,cleaned,btc-4h.csv,18876,18876,18846,0,30
2,ETHUSDT,eth,1d,raw\eth-1d-RAW.csv,eth-1d.csv,cleaned,eth-1d.csv,3149,3149,3119,0,30
3,ETHUSDT,eth,4h,raw\eth-4h-RAW.csv,eth-4h.csv,cleaned,eth-4h.csv,18691,18691,18661,0,30
4,BNBUSDT,bnb,1d,raw\bnb-1d-RAW.csv,bnb-1d.csv,cleaned,bnb-1d.csv,3068,3068,3038,0,30
5,BNBUSDT,bnb,4h,raw\bnb-4h-RAW.csv,bnb-4h.csv,cleaned,bnb-4h.csv,18392,18392,18362,0,30
6,XRPUSDT,xrp,1d,raw\xrp-1d-RAW.csv,xrp-1d.csv,cleaned,xrp-1d.csv,2889,2889,2859,0,30
7,XRPUSDT,xrp,4h,raw\xrp-4h-RAW.csv,xrp-4h.csv,cleaned,xrp-4h.csv,16969,16969,16939,0,30
8,DOGEUSDT,doge,1d,raw\doge-1d-RAW.csv,doge-1d.csv,cleaned,doge-1d.csv,2462,2462,2432,0,30
9,DOGEUSDT,doge,4h,raw\doge-4h-RAW.csv,doge-4h.csv,cleaned,doge-4h.csv,14767,14767,14737,0,30


In [12]:
# Финальный контроль наличия файлов
final_rows = []
for t in tasks:
    raw_exists = Path(t["raw_csv"]).exists()
    clean_exists = Path(t["cleaned_csv"]).exists()
    final_rows.append({**t, "raw_exists": raw_exists, "clean_exists": clean_exists})

pd.DataFrame(final_rows)


,symbol,active,interval,raw_csv,cleaned_csv,raw_exists,clean_exists
0,BTCUSDT,btc,1d,raw\btc-1d-RAW.csv,btc-1d.csv,True,True
1,BTCUSDT,btc,4h,raw\btc-4h-RAW.csv,btc-4h.csv,True,True
2,ETHUSDT,eth,1d,raw\eth-1d-RAW.csv,eth-1d.csv,True,True
3,ETHUSDT,eth,4h,raw\eth-4h-RAW.csv,eth-4h.csv,True,True
4,BNBUSDT,bnb,1d,raw\bnb-1d-RAW.csv,bnb-1d.csv,True,True
5,BNBUSDT,bnb,4h,raw\bnb-4h-RAW.csv,bnb-4h.csv,True,True
6,XRPUSDT,xrp,1d,raw\xrp-1d-RAW.csv,xrp-1d.csv,True,True
7,XRPUSDT,xrp,4h,raw\xrp-4h-RAW.csv,xrp-4h.csv,True,True
8,DOGEUSDT,doge,1d,raw\doge-1d-RAW.csv,doge-1d.csv,True,True
9,DOGEUSDT,doge,4h,raw\doge-4h-RAW.csv,doge-4h.csv,True,True
